In [ ]:
import os
import pandas as pd
import librosa
import torch
import numpy as np
import matplotlib.pyplot as plt
from numpy import ndarray
import random
import glob
import matplotlib.pyplot as plt

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# F0 inconsistency

In [ ]:
def load_real_audio(name):
    path = os.path.join(real_dir, f"{name}.flac")
    x, _ = librosa.load(path, sr=None)
    return x
    

class STFTWindowParams:
    def __init__(self, sample_rate=16000, win_sec=0.032, hop_sec=0.016):
        self.sample_rate = sample_rate
        self.win_sec = win_sec
        self.hop_sec = hop_sec

        self.update()

    def update(self):
        self.win_size = int(self.win_sec * self.sample_rate)
        self.hop_size = int(self.hop_sec * self.sample_rate)
        self.fft_size = self.next_pow_2(self.win_size)

    @staticmethod
    def next_pow_2(x):
        return 2 ** (x - 1).bit_length()


def extract_f0(waveform,
              fs, win_size, hop_size,
              fmin: int = 70, fmax: int = 400) -> ndarray:

    x = np.copy(waveform)
    x = librosa.effects.preemphasis(x)
    autocorr_win_len = win_size // 2

    f0_freqs, _, _ = librosa.pyin(
        x, sr=fs,
        frame_length=win_size, hop_length=hop_size, center=True,
        win_length=autocorr_win_len, fmin=fmin, fmax=fmax, n_thresholds=7
    )

    return f0_freqs

    
def flatten_f0(f0_traj, strength=0.9, p=0.8):

    f0_mod = f0_traj.copy()
    voiced = ~np.isnan(f0_mod)

    f0_ref = np.nanmedian(f0_mod)
    mask = (np.random.rand(len(f0_mod)) < p) & voiced

    f0_mod[mask] = (
            (1.0 - strength) * f0_mod[mask]
            + strength * f0_ref
    )

    return f0_mod

    
def pitch_shift_f0(f0_traj, shift_factor):
    f0_mod = f0_traj.copy()
    voiced = ~np.isnan(f0_mod)
    f0_mod[voiced] *= shift_factor
    return f0_mod

    
def partial_pitch_shift_f0(f0_traj, shift_factor, p=0.5):
    f0_mod = f0_traj.copy()
    voiced = ~np.isnan(f0_mod)
    mask = (np.random.rand(len(f0_mod)) < p) & voiced
    f0_mod[mask] *= shift_factor
    return f0_mod


def apply_f0_mod(mod='partial'):
    
    if mod == 'partial':
        # --- partial pitch shift ---
        shift = random.choice([
            random.uniform(0.3, 0.8),
            random.uniform(1.5, 2)
        ])
        f0_traj_mod = partial_pitch_shift_f0(
            f0_traj,
            shift_factor=shift,
            p=random.uniform(0.3, 0.8)
        )
    
    elif mod == 'flat':
        # --- f0 flattening ---
        f0_traj_mod = flatten_f0(
            f0_traj,
            # strength=random.uniform(0.6, 0.9),
            p=random.uniform(0.5, 0.9)
        )
    
    else:
        # --- full pitch shift ---
        shift = random.choice([
            random.uniform(0.3, 0.8),
            random.uniform(1.5, 2)
        ])
        f0_traj_mod = pitch_shift_f0(
            f0_traj,
            shift_factor=shift
        )

    return f0_traj_mod
    
'''    
def to_TF_plane(traj, fs, n_fft):
    frequency_resolution = fs / n_fft

    time_idxs = np.argwhere(np.isnan(traj.T) == False)
    freq_idxs = (traj[time_idxs] / frequency_resolution).astype(int)

    out_contour_matrix = np.zeros((n_fft // 2 + 1, traj.shape[0]))
    out_contour_matrix[freq_idxs, time_idxs] = np.ones(time_idxs.shape)

    return out_contour_matrix

def extract_mag(x):
    
    stft_params = STFTWindowParams()
    window = torch.hann_window(stft_params.fft_size, device='cpu')
    
    X = torch.stft(
        x,
        n_fft=stft_params.fft_size,
        hop_length=stft_params.hop_size,
        win_length=stft_params.win_size,
        window=window,
        center=True,
        return_complex=True
    )
    
    mag = torch.abs(X)
    phase = torch.angle(X)
    
    return mag, phase

def mag_to_db(mag):
    return torchaudio.functional.amplitude_to_DB(
        mag,
        multiplier=20.0,  
        amin=1e-10,
        db_multiplier=1.0
    )


X, _ = extract_mag(x)

mask_old = gaussian_filter(to_TF_plane(f0_traj, fs, stft_params.fft_size), sigma=1)[None, :, :]
mask_new = gaussian_filter(to_TF_plane(f0_traj_mod, fs, stft_params.fft_size), sigma=1)[None, :, :]

att = 1  # attenuation factor, [0, 1]
emp = 2  # emphasis factor [>= 1]
mask_old = torch.from_numpy(mask_old).float().to(mag.device)
mask_new = torch.from_numpy(mask_new).float().to(mag.device)
mag_new = mag * (1.0 - att * mask_old)
mag_new = mag_new * (1.0 + emp * mask_new)

mag_new = torch.clamp(mag_new, min=1e-8)
mag = mag_new.squeeze()
label = 1

mag = mag_to_db(mag)
'''

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

real_dir = '/Database/ASVspoof5/flac_D'

all_flacs = glob.glob(os.path.join(real_dir, '*.flac'))
selected_files = random.sample(all_flacs, 10)

In [ ]:
mod = 'flat'

In [ ]:
fs = 16000
stft_params = STFTWindowParams()
rows = []

nan_count = 0
total = len(selected_files)

for f in selected_files:
    name = os.path.splitext(f)[0]
    x = load_real_audio(name)
    
    f0_traj = extract_f0(
        x,
        fs=stft_params.sample_rate,
        win_size=stft_params.win_size,
        hop_size=stft_params.hop_size
    )
    
    if np.all(np.isnan(f0_traj)):
        f0_traj_mod = None
        nan_count += 1
    else:
        f0_traj_mod = apply_f0_mod(mod=mod)
        
    rows.append({
        "name": os.path.splitext(os.path.basename(name))[0],
        "f0_traj": f0_traj,
        "f0_traj_mod": f0_traj_mod
    })

print(f"{nan_count}/{total} tracks have all-NaN F0 trajectories")

df = pd.DataFrame(rows)
df

In [ ]:
df_valid = df[df["f0_traj_mod"].notnull()].reset_index(drop=True)

n_tracks = len(df_valid)

fig, axes = plt.subplots(
    n_tracks, 2,
    figsize=(18, 4 * n_tracks),
    sharex=False
)

if n_tracks == 1:
    axes = axes[np.newaxis, :]

for i, row in df_valid.iterrows():
    f0 = row["f0_traj"]
    f0_mod = row["f0_traj_mod"]

    n_frames = len(f0)
    time = np.arange(n_frames) * (stft_params.hop_size / stft_params.sample_rate)

    # Original
    axes[i, 0].plot(time, f0)
    axes[i, 0].set_title(f"{row['name']} – Original F0")
    axes[i, 0].set_ylim(0, 1000)
    axes[i, 0].set_ylabel("Frequency (Hz)")

    # Modified
    axes[i, 1].plot(time, f0_mod)
    axes[i, 1].set_title("Modified F0")
    axes[i, 1].set_ylim(0, 1000)

    for j in range(2):
        axes[i, j].set_xlabel("Time (s)")
        axes[i, j].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
